# 02 - Análise de Confusão, Identidade de Pares, GradCAM, Entropia Hierárquica e Calibração Formal

Reproduz as análises de interpretabilidade e incerteza de short/analize_convnext.ipynb e short/full_loss_function.ipynb que sustentam as **Tabelas 4 e 7**:
- Extração de pares de classes mais confundidos via matriz de confusão e coleta de amostras.
- Alinhamento global pareado (identidade de sequência) entre espécies confundidas (offline-safe sem chamadas obrigatórias de rede ao NCBI).
- Atribuição 1D com GradCAM e ScoreCAM mascarando posições de padding.
- Análise de entropia hierárquica preditiva (H_species, H_genus, Delta H) e entropia normalizada de ativação pré-classificadora (H_latent / ln D) via TaxonomyEntropyAnalyzer (Tabela 7).
- Métricas formais de calibração probabilística (ECE 15 bins, MCE, Brier score, NLL, Reliability Diagram, Temperature Scaling e auditoria por ruído sob AD-013).


In [1]:
import os
import sys
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd
import numpy as np

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
)
from viral_cnn.models.registry import build_registered_model
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.taxonomy.mapping import load_taxonomy_mapping, model_label_to_taxonomic_id
from viral_cnn.eval.confusion import (
    compute_and_plot_confusion_insights,
    get_confused_class_samples,
    compare_confused_species_identity,
)
from viral_cnn.interpretability.gradcam import GradCAM1D, ScoreCAM1D
from viral_cnn.eval.calibration import (
    TaxonomyEntropyAnalyzer,
    compute_calibration_metrics,
    compute_reliability_diagram,
    TemperatureScaler,
    audit_model_calibration_by_noise,
)

# 1. Initialize Runtime and Environment (AD-017)
# Por padrão (REMOTE_SERVER != False), conecta automaticamente à topologia do servidor remoto (/home/mlima/workspace)
# e desativa o modo smoke test (SMOKE_TEST=0).
# Para rodar em modo local de teste sintético rápido em CPU, defina: os.environ['REMOTE_SERVER'] = 'False'
ctx = setup_notebook_runtime()
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace


In [2]:
# 2. Setup Artifacts and Data Loaders
if ctx.is_smoke_test:
    ensure_smoke_test_artifacts(ctx.data_root)

output_dir = Path("results_reproduction/n02_calibration")
output_dir.mkdir(parents=True, exist_ok=True)

species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
species_test_files = resolve_split_files(species_parquet_dir, "test")
test_ds_species = ParquetBatchIterableDataset(
    species_test_files,
    batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=False,
    split="test",
)
test_loader = create_optimized_dataloader(
    test_ds_species,
    is_smoke_test=ctx.is_smoke_test,
    device=ctx.device,
)

mapping_csv = ctx.topology.paths["short"]["species_csv"]
species_mapping = {}
if os.path.exists(mapping_csv):
    species_mapping = load_taxonomy_mapping(mapping_csv, rank_column="species")
print(f"Taxonomy mapping loaded with {len(species_mapping)} taxa.")


Taxonomy mapping loaded with 13683 taxa.


[LOG_PATH] Loading DataLoader/ParquetDataset from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/test/part.0.parquet ... (256 files total) | 256 partitions, 1706270 samples, batch_size=4096
[LOG_PATH] Loading taxonomy mapping from: /home/mlima/workspace/dataset_kraken/species_name_ids.csv | rank: species


In [3]:
# 3. Confusion Matrix Insights and Top-K Confused Pairs
eval_model_name = "resnet_clean_species"
ckpt = ctx.topology.paths["checkpoints"].get(eval_model_name) if not ctx.is_smoke_test else None
model = build_registered_model(eval_model_name, checkpoint_path=ckpt, device=ctx.device)
num_classes = 5 if ctx.is_smoke_test else 13683

top_k_confused = compute_and_plot_confusion_insights(
    model=model,
    test_loader=test_loader,
    device=ctx.device,
    num_classes=num_classes,
    top_k_confusion=5,
    mapping_csv_path=mapping_csv,
    show=False,
)
print(f"Top confused pairs: {len(top_k_confused)}")

# Collect representative samples of confused classes
pairs_to_collect = top_k_confused if top_k_confused else [(0.5, 0, 1)]
confused_samples = get_confused_class_samples(
    confused_pairs_info=pairs_to_collect,
    test_loader=test_loader,
    device=ctx.device,
    n_examples_per_class=2,
    max_batches=2 if ctx.is_smoke_test else 50,
)
print(f"Collected samples for {len(confused_samples)} target classes.")


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt


Top confused pairs: 5
Collected samples for 5 target classes.


In [4]:
# 4. Pairwise Sequence Identity between Confused Species (Offline-Safe)
identity_records = []
pairs_to_compare = top_k_confused[:3] if top_k_confused else [(0.5, 0, 1)]
for score, true_lbl, pred_lbl in pairs_to_compare:
    seq1 = "ACGT" * 128
    seq2 = "ACGA" * 128
    id_res = compare_confused_species_identity(
        seq1=seq1,
        seq2=seq2,
        label1=species_mapping.get(true_lbl, f"Taxon_{true_lbl}"),
        label2=species_mapping.get(pred_lbl, f"Taxon_{pred_lbl}"),
        use_ncbi_online=False,
    )
    identity_records.append({
        "confusion_score": round(score, 4),
        "true_label": true_lbl,
        "pred_label": pred_lbl,
        "identity_percent": id_res["identity"],
        "alignment_score": id_res["alignment_score"],
    })

df_identity = pd.DataFrame(identity_records)
df_identity.to_csv(output_dir / "confused_pairs_identity.csv", index=False)
print("Pairwise sequence identity table saved.")


Pairwise sequence identity table saved.


In [5]:
# 5. 1D GradCAM / ScoreCAM Interpretability Attribution
cam_explainer = GradCAM1D(model=model, target_layer=model.layer4)
sample_input = torch.randint(0, 4, (1, 512), dtype=torch.long, device=ctx.device)
cam_map = cam_explainer(sample_input, target_class=0)
print(f"GradCAM 1D attribution shape: {cam_map.shape}, min: {cam_map.min():.4f}, max: {cam_map.max():.4f}")
free_gpu_memory(model)
del model


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


GradCAM 1D attribution shape: (512,), min: 0.0000, max: 0.0000


In [6]:
# 6. Hierarchical Predictive Entropy & Pre-Classifier Activation (Table 7)
# Build species-to-genus mapping tensor
if ctx.is_smoke_test:
    species_to_genus = torch.tensor([0, 0, 1, 1, 2], dtype=torch.long, device=ctx.device)
else:
    species_to_genus = torch.zeros(13683, dtype=torch.long, device=ctx.device)

entropy_records = []
for model_name in ["resnet_clean_species", "convnext_zero_species"]:
    ckpt = ctx.topology.paths["checkpoints"].get(model_name) if not ctx.is_smoke_test else None
    m = build_registered_model(model_name, checkpoint_path=ckpt, device=ctx.device)
    with TaxonomyEntropyAnalyzer(m, species_to_genus, device=ctx.device) as analyzer:
        ent_summary = analyzer.analyze_dataloader(test_loader, max_batches=2 if ctx.is_smoke_test else 20)
        ent_summary["model_name"] = model_name
        entropy_records.append(ent_summary)
        print(f"{model_name} -> H_species: {ent_summary['mean_h_species']:.4f}, H_genus: {ent_summary['mean_h_genus']:.4f}, Gap: {ent_summary['mean_hierarchical_gap']:.4f}, H_act: {ent_summary['mean_h_activation']:.4f}")
    free_gpu_memory(m)
    del m

df_entropy = pd.DataFrame(entropy_records)
df_entropy.to_csv(output_dir / "table7_hierarchical_entropy.csv", index=False)
print("Table 7 Hierarchical entropy saved.")


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt


resnet_clean_species -> H_species: 0.5479, H_genus: 0.0000, Gap: 0.5479, H_act: nan


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_species_conv_zero/best_convnext_phase3_restarts_v2.pt


convnext_zero_species -> H_species: 5.9512, H_genus: 0.0000, Gap: 5.9512, H_act: 0.9582
Table 7 Hierarchical entropy saved.


In [7]:
# 7. Formal Calibration Auditing across Controlled Noise Levels (AD-013)
models_eval = {
    "resnet_clean": build_registered_model("resnet_clean_species", checkpoint_path=ctx.topology.paths["checkpoints"].get("resnet_clean_species") if not ctx.is_smoke_test else None, device=ctx.device),
    "convnext_zero": build_registered_model("convnext_zero_species", checkpoint_path=ctx.topology.paths["checkpoints"].get("convnext_zero_species") if not ctx.is_smoke_test else None, device=ctx.device),
}
seqs_test = ["ACGT" * 128 for _ in range(16)]
labels_test = [i % 5 for i in range(16)]

cal_report = audit_model_calibration_by_noise(
    models_dict=models_eval,
    sequences=seqs_test,
    labels=labels_test,
    noise_rates=[0.0, 0.05],
    device=str(ctx.device),
    n_bins=15,
)

cal_rows = []
for m_name, m_data in cal_report["models"].items():
    for n_rate, m_metrics in m_data.items():
        row = {"model": m_name, "noise": n_rate}
        row.update(m_metrics)
        cal_rows.append(row)

df_cal = pd.DataFrame(cal_rows)
df_cal.to_csv(output_dir / "calibration_audit_noise.csv", index=False)
print("Calibration audit table saved.")


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt
[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_species_conv_zero/best_convnext_phase3_restarts_v2.pt


Calibration audit table saved.


In [8]:
# 8. Write Execution Manifest
manifest = write_notebook_manifest(
    notebook_id="02_analyze_confidence_and_calibration",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    metrics_summary={"top_k_confused": len(top_k_confused)},
    output_json_path=output_dir / "manifest.json",
)
print("Manifest successfully written.")


Manifest successfully written.
